<a href="https://colab.research.google.com/github/Leo278V/Final-Assignment-PDS/blob/Final-Assignment-V2/Final-Assignment-V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [114]:
# Rule Based Approach

In [115]:
import pandas as pd
import numpy as np


In [116]:
# Data Path CSV Files

data_path_department = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/department-v2.csv"
df_department = pd.read_csv(data_path_department)
df_department.head()

,text,label
0,Adjoint directeur communication,Marketing
1,Advisor Strategy and Projects,Project Management
2,Beratung & Projekte,Project Management
3,Beratung & Projektmanagement,Project Management
4,Beratung und Projektmanagement kommunale Partner,Project Management


In [117]:
data_path_seniority = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/seniority-v2.csv"
df_seniortiy = pd.read_csv(data_path_seniority)
df_seniortiy.head()

,text,label
0,Analyst,Junior
1,Analyste financier,Junior
2,Anwendungstechnischer Mitarbeiter,Junior
3,Application Engineer,Senior
4,Applications Engineer,Senior


In [118]:
# Data Path Annotated LinkedIN Profiles
data_path_profiles = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/linkedin_experience_annotated.csv"
df_profiles = pd.read_csv(data_path_profiles)
df_profiles.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,NaN,ACTIVE,Other,Management,0
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,NaN,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,NaN,ACTIVE,Other,Management,0
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0


In [119]:
# Filtering for active
df_active = df_profiles[df_profiles["status"] == "ACTIVE"].copy()

# Drop Department & seniority for later evaluation

df_active_no_labels = df_active.drop(
    columns=["department", "seniority","linkedin"],
    errors="ignore"
)
df_active_no_labels.head()

,organization,position,startDate,endDate,status,person_id
0,Depot4Design GmbH,Prokurist,2019-08,NaN,ACTIVE,0
1,Depot4Design GmbH,CFO,2019-07,NaN,ACTIVE,0
2,Depot4Design GmbH,Betriebswirtin,2019-07,NaN,ACTIVE,0
3,Depot4Design GmbH,Prokuristin,2019-07,NaN,ACTIVE,0
4,Depot4Design GmbH,CFO,2019-07,NaN,ACTIVE,0


In [126]:
# Creating Dicitionaries

#Seniority
seniority_dict = (
    df_seniortiy
    .dropna(subset=["text", "label"])
    .assign(keyword=lambda x: x["text"].str.lower())
    .set_index("keyword")["label"]
    .to_dict()
)

#Department
department_dict = (
    df_department
    .dropna(subset=["text", "label"])
    .assign(keyword=lambda x: x["text"].str.lower())
    .set_index("keyword")["label"]
    .to_dict()
)


In [127]:
# Matching Dictionaries function

def predict_labels(df, department_dict, seniority_dict):
    positions = df["position"].fillna("").str.lower()
    return pd.DataFrame({
        "pred_department": positions.apply(lambda x: dict_match(x, department_dict)),
        "pred_seniority": positions.apply(lambda x: dict_match(x, seniority_dict)),
    })



In [132]:
# Application on Annotated CVs

predictions = predict_labels(df_active_no_labels, department_dict, seniority_dict)

df_active_no_labels["pred_department"] = predictions["pred_department"]
df_active_no_labels["pred_seniority"] = predictions["pred_seniority"]

In [133]:
#Successful Classification
dept_coverage = df_active_no_labels["pred_department"].notna().mean()
sen_coverage = df_active_no_labels["pred_seniority"].notna().mean()

print(f"Department Coverage: {dept_coverage:.2%}")
print(f"Seniority Coverage: {sen_coverage:.2%}")

Department Coverage: 32.74%
Seniority Coverage: 61.48%


In [134]:
#Accuracy with assigned labels for Department
# Merge predicted department into df_active for evaluation
df_active_merged_dept = df_active.merge(
    df_active_no_labels[['person_id', 'pred_department']],
    on='person_id',
    how='left'
)

dept_eval = df_active_merged_dept.dropna(
    subset=["department", "pred_department"]
)

dept_accuracy = (
    dept_eval["department"] == dept_eval["pred_department"]
).mean()

print(f"Department Accuracy: {dept_accuracy:.2%}")

Department Accuracy: 39.38%


In [135]:
#Accuracy with assigned labels for Seniority
# Merge predicted seniority into df_active for evaluation
df_active_merged_sen = df_active.merge(
    df_active_no_labels[['person_id', 'pred_seniority']],
    on='person_id',
    how='left'
)

sen_eval = df_active_merged_sen.dropna(
    subset=["seniority", "pred_seniority"]
)

sen_accuracy = (
    sen_eval["seniority"] == sen_eval["pred_seniority"]
).mean()

print(f"Seniority Accuracy: {sen_accuracy:.2%}")

Seniority Accuracy: 55.59%
